[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_37_vLLM_Serving_Your_Own_Model.ipynb)

# Lesson 37 — Track 3 · vLLM: Serve Your Own LLM

**Phase 4 → Track 3 (Self-hosted & Fine-tuning) · Lesson 1 of 5**

> *"An API call costs $0.003/1K tokens at 3 AM. Your own T4 GPU costs $0.35/hr and handles 1,500 tok/s. The math changes fast at volume."*

## Track 3 Roadmap
| Lesson | Topic | Why it matters |
|--------|-------|----------------|
| **L37 ← you are here** | vLLM + serving your own model | Cost control, latency, privacy |
| L38 | QLoRA fine-tuning on real data | Domain-specific expertise |
| L39 | DPO/ORPO preference tuning | Align behavior without full RLHF |
| L40 | Distillation — big model → small | Deploy cheap expert-level models |
| L41 | Model merging (SLERP/TIES) | Combine capabilities without retraining |

## What you'll build
1. A **vLLM inference engine** running `Qwen2.5-1.5B-Instruct` on Colab T4
2. Queries via the **OpenAI-compatible client** (drop-in swap from the API)
3. A **throughput benchmark**: vLLM vs. naive HuggingFace sequential generate
4. The server wired into your **L31 reliability spine** as Tier 0 (cheapest tier)

## Prerequisites
- **GPU runtime** in Colab: Runtime → Change runtime type → T4 GPU
- `ANTHROPIC_API_KEY` in Colab Secrets (for Tier 1 Haiku fallback demo)


In [ ]:
# ── §0 Setup ──────────────────────────────────────────────────────────────────
import subprocess, sys, os, time

# 1. Check GPU availability
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ GPU: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("⚠️  No GPU detected.")
    print("   Go to: Runtime → Change runtime type → T4 GPU")
    print("   GPU cells will show code but use stub outputs.")
    HAS_GPU = False

# 2. Load API key (for Haiku fallback tier)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ Anthropic API key loaded from Colab Secrets")
except Exception:
    os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-REPLACE_ME")
    print("ℹ️  Add ANTHROPIC_API_KEY to Colab Secrets for the fallback demo")

# 3. Install packages
print("\n📦 Installing packages...")
if HAS_GPU:
    print("   (vLLM builds CUDA kernels — first install takes ~3 min)")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "vllm", "openai", "anthropic", "transformers",
                    "accelerate", "--quiet"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "openai", "anthropic", "--quiet"], check=True)

import anthropic
from openai import OpenAI
print("\n✅ All imports ready")


## §1 Why Self-Host? — The Decision Framework

Three forces drive the self-hosting decision: **cost**, **latency**, and **privacy**.

### Cost math (approximate 2025 numbers)

| Option | Input cost | TTFT | Privacy |
|--------|-----------|------|---------|
| Anthropic Haiku 4.5 (API) | $0.80 / MTok | 300–700 ms | Off-prem |
| Anthropic Sonnet 4.5 (API) | $3.00 / MTok | 300–700 ms | Off-prem |
| Qwen-2.5-7B on A10G (self) | ~$0.05 / MTok equiv | 50–150 ms | On-prem |
| Qwen-2.5-1.5B on T4 (self) | ~$0.01 / MTok equiv | 30–80 ms | On-prem |

The "MTok equivalent" for self-hosted is GPU-hours amortized over throughput.

### When self-hosting wins
- **Volume > ~10M tokens/day** — cost savings dominate
- **Latency SLA < 100 ms** — no network hop to API
- **Regulated data** — HIPAA, SOC2, sovereign cloud requirements
- **Fine-tuned specialization** — domain data that API models can't absorb

### When API wins
- **Bursty/unpredictable traffic** — no idle GPU costs at 3 AM
- **Prototyping or < 1M tokens/day**
- **Frontier reasoning** — GPT-4o / Claude Sonnet-class capability
- **No infra team** — API is someone else's ops problem

### The agent engineer's practical rule
> Use **API for the orchestrator brain** (needs frontier reasoning for planning),  
> **self-host for the worker bees** (summarization, classification, extraction — high volume, latency-sensitive).


In [ ]:
# ── §1a Cost Comparison ───────────────────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class PricingOption:
    name: str
    input_per_mtok: float    # $ per million input tokens
    output_per_mtok: float   # $ per million output tokens
    ttft_ms: float           # typical time-to-first-token

options = [
    PricingOption("Haiku 4.5 (API)",      0.80,  4.00, 400),
    PricingOption("Sonnet 4.5 (API)",     3.00, 15.00, 500),
    PricingOption("Qwen-2.5-7B (self)",   0.05,  0.05,  90),
    PricingOption("Qwen-2.5-1.5B (self)", 0.01,  0.01,  50),
]

# Scenario: 10M input + 2M output tokens per day
INPUT_TOKENS_DAY  = 10_000_000
OUTPUT_TOKENS_DAY =  2_000_000

print(f"Daily cost @ {INPUT_TOKENS_DAY/1e6:.0f}M input + {OUTPUT_TOKENS_DAY/1e6:.0f}M output tokens\n")
print(f"{'Option':<28} {'Daily cost':>12}  {'Monthly cost':>14}  {'TTFT (ms)'}")
print("-" * 70)
for o in options:
    daily  = (INPUT_TOKENS_DAY  / 1e6 * o.input_per_mtok +
              OUTPUT_TOKENS_DAY / 1e6 * o.output_per_mtok)
    monthly = daily * 30
    print(f"{o.name:<28} ${daily:>11.2f}  ${monthly:>13.2f}  {o.ttft_ms:.0f}")

# 💡 EXPERIMENT: Change INPUT_TOKENS_DAY to 1_000_000 (1M).
#    At what volume does the self-hosted T4 ($0.35/hr GPU rental) break even vs Haiku?


## §2 vLLM Architecture — Why It's So Fast

vLLM's speed comes from two innovations that address the core bottlenecks in LLM serving.

### Bottleneck 1: KV Cache Memory Fragmentation

During generation, each token attends to all previous tokens via **Key-Value matrices** (the KV cache). Naive HuggingFace pre-allocates a contiguous block per sequence:

```
Request A (max 512 tok): [KV KV KV KV _ _ _ _ _ _ _ _ ]  ← 60% wasted
Request B (max 512 tok): [KV KV _ _ _ _ _ _ _ _ _ _ _ ]  ← 85% wasted
Free memory used for "potential" tokens that never arrive
```

**PagedAttention** (Kwon et al., 2023) treats KV cache like OS virtual memory — paged and non-contiguous:

```
Physical KV blocks: [B:1][B:2][A:1][A:3][B:3][A:2][FREE][FREE]
Logical mapping:    A → [A:1, A:2, A:3]   B → [B:1, B:2, B:3]
```

Result: near-zero fragmentation, ~2-4× more sequences in the same VRAM.

### Bottleneck 2: Batch Padding Waste

Naive batching runs until the *longest* sequence finishes — shorter sequences waste GPU cycles:

```
Naive batch (all must wait for C):
  Step 1-8:  [A running][B running][C running]
  Step 9-20: [A DONE   ][B DONE   ][C running]  ← A and B slots sit idle
```

**Continuous batching**: finished sequences are immediately evicted, new ones injected mid-batch:

```
Step 1-4:  [A][B][C]
Step 5:    [A][D][C]    ← B finished step 4, D starts immediately  
Step 9:    [E][D][C]    ← A finishes, E starts immediately
```

### Combined result

| Method | Throughput | Memory use | TTFT |
|--------|-----------|------------|------|
| HF `.generate()` sequential | 1× | Poor (fixed alloc) | High |
| HF batched (naive) | ~2-3× | Poor | Medium |
| **vLLM (PagedAttention + continuous batching)** | **~10-24×** | **Excellent** | **Low** |

The exact speedup depends on batch size and sequence length variance.  
Single request? ~2-3× (mainly PagedAttention). 50 concurrent? ~15-20× (continuous batching shines).


## §3 VRAM Math — Picking the Right Model

Before loading any model, calculate whether it fits. The formula:

```
Total VRAM = weight_bytes + kv_cache_bytes + overhead

weight_bytes  = params_billions × 1e9 × bytes_per_param
  FP16/BF16 → 2 bytes/param
  INT8      → 1 byte/param  
  INT4      → 0.5 bytes/param (AWQ/GPTQ quantization)

kv_cache_bytes = 2 × n_layers × n_heads × head_dim × seq_len × batch_size × 2
  (2 for K and V, last 2 for BF16)

overhead ≈ 1-2 GB (CUDA kernels, activations, gradients)
```

**Colab T4 = 16 GB VRAM**. Leave ~1 GB headroom for overhead.


In [ ]:
# ── §3a VRAM Calculator ───────────────────────────────────────────────────────
def vram_estimate(
    params_b: float,          # model billions of parameters
    precision: str = "bf16",  # bf16, fp16, int8, int4
    batch_size: int = 8,
    seq_len: int = 2048,
    n_layers: int = 28,
    n_heads: int = 16,
    head_dim: int = 128,
) -> dict:
    bpp = {"bf16": 2, "fp16": 2, "int8": 1, "int4": 0.5}[precision]
    weights_gb  = params_b * 1e9 * bpp / 1e9
    kv_gb       = (2 * n_layers * n_heads * head_dim * seq_len * batch_size * 2) / 1e9
    total_gb    = weights_gb + kv_gb + 1.0  # +1 overhead
    return {
        "weights_gb":  round(weights_gb, 2),
        "kv_cache_gb": round(kv_gb, 2),
        "total_gb":    round(total_gb, 2),
        "fits_t4":     total_gb < 15.5,
    }

# Models we might run on Colab
MODELS = [
    # (display_name,      params_b, n_layers, n_heads, head_dim)
    ("Qwen2.5-0.5B",      0.5,  24, 14,  64),
    ("Qwen2.5-1.5B",      1.5,  28, 16, 128),
    ("Qwen2.5-7B",        7.0,  28, 28, 128),
    ("Llama-3.1-8B",      8.0,  32, 32, 128),
    ("Mistral-7B",        7.0,  32, 32, 128),
]

print(f"{'Model':<20} {'Prec':<7} {'Weights':>8} {'KV cache':>10} {'Total':>8}  T4 OK?")
print("-" * 65)
for name, pb, nl, nh, hd in MODELS:
    for prec in ["bf16", "int4"]:
        r = vram_estimate(pb, prec, batch_size=8, seq_len=2048,
                         n_layers=nl, n_heads=nh, head_dim=hd)
        ok = "✅" if r["fits_t4"] else "❌"
        print(f"{name:<20} {prec:<7} {r['weights_gb']:>7.1f}G {r['kv_cache_gb']:>9.2f}G "
              f"{r['total_gb']:>7.2f}G  {ok}")
    print()

print("We'll use Qwen2.5-1.5B in BF16 → ~4 GB total, leaves 12 GB for KV cache buffer")
print("This gives us comfortable headroom for batch_size=32 at seq_len=2048")

# 💡 EXPERIMENT: Change batch_size to 32 and seq_len to 4096.
#    Which models still fit on T4 in BF16? Which need int4?


## §4 vLLM Python API — Batch Inference

The simplest way to use vLLM is the `LLM` class. No server setup needed:

```python
from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", dtype="bfloat16")
outputs = llm.generate(prompts, SamplingParams(temperature=0.7, max_tokens=200))
```

Key `LLM` constructor args:
- `dtype` — `"bfloat16"` for modern GPUs (A10G, H100), `"float16"` for V100/older T4
- `max_model_len` — cap the context window to save KV cache VRAM
- `gpu_memory_utilization` — fraction of VRAM for KV cache (default 0.9, use 0.85 on T4)
- `quantization` — `"awq"` or `"gptq"` for pre-quantized model variants


In [ ]:
# ── §4a Load the Model ────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

if HAS_GPU:
    from vllm import LLM, SamplingParams
    import torch

    print(f"⏳ Loading {MODEL_ID}...")
    print("   (First run downloads ~3 GB from HuggingFace Hub)")

    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        max_model_len=4096,          # cap context to save KV cache VRAM
        gpu_memory_utilization=0.85, # leave 15% headroom for OS + CUDA kernels
    )
    print("✅ Model loaded\n")

    allocated_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb  = torch.cuda.memory_reserved()  / 1e9
    print(f"   VRAM allocated: {allocated_gb:.2f} GB")
    print(f"   VRAM reserved:  {reserved_gb:.2f} GB")
    print(f"   KV cache headroom: {16.0 - reserved_gb:.2f} GB")
else:
    llm = None
    print("(Stub mode — no GPU. Code shown for reference.)")


In [ ]:
# ── §4b Batch Generate — the core vLLM superpower ────────────────────────────
# vLLM processes ALL prompts simultaneously with continuous batching.
# You submit N prompts in one call; they're scheduled optimally across GPU cores.

PROMPTS = [
    "Explain gradient descent in one sentence.",
    "What is the transformer attention mechanism?",
    "Why is layer normalization important in deep learning?",
    "Describe the vanishing gradient problem briefly.",
    "What is the difference between RNN and LSTM?",
    "Explain what token embeddings represent in NLP.",
    "What does 'context window' mean for an LLM?",
    "Why do transformers outperform RNNs on long sequences?",
]

if HAS_GPU:
    sampling = SamplingParams(temperature=0.7, max_tokens=80, top_p=0.9)

    t0 = time.time()
    outputs = llm.generate(PROMPTS, sampling)
    elapsed = time.time() - t0

    total_out_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    total_in_tokens  = sum(len(o.prompt_token_ids)    for o in outputs)

    print(f"✅ {len(PROMPTS)} responses in {elapsed:.2f}s")
    print(f"   Input tokens:  {total_in_tokens}")
    print(f"   Output tokens: {total_out_tokens}")
    print(f"   Throughput:    {total_out_tokens/elapsed:.0f} output tok/s\n")

    # Show first 2
    for prompt, out in zip(PROMPTS[:2], outputs[:2]):
        print(f"Q: {prompt}")
        print(f"A: {out.outputs[0].text.strip()}")
        print()
else:
    print("(Stub) 8 responses in 0.8s | ~900 output tokens | ~1125 tok/s")
    print()
    print("Q: Explain gradient descent in one sentence.")
    print("A: Gradient descent iteratively adjusts model weights by stepping in the")
    print("   direction that minimally reduces the loss function.")

# 💡 EXPERIMENT: Double PROMPTS to 16 entries.
#    Notice throughput (tok/s) goes UP — that's continuous batching at work.
#    With sequential HF, doubling the prompts doubles the time.


## §5 vLLM OpenAI-Compatible Server

For production, you don't call `LLM.generate()` directly — you run a **server**.

vLLM ships an OpenAI-compatible REST API. The critical advantage: **your existing code works unchanged**. Just change `base_url`.

```python
# Before (Anthropic API)
client = anthropic.Anthropic()
response = client.messages.create(model="claude-haiku-4-5", ...)

# After (self-hosted vLLM) — openai client, same interface
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
response = client.chat.completions.create(model="Qwen/Qwen2.5-1.5B-Instruct", ...)
```

**Endpoints provided:**
- `GET  /v1/models` — list available models
- `POST /v1/chat/completions` — chat inference (streaming supported)
- `POST /v1/completions` — raw text completion
- `POST /v1/embeddings` — embedding vectors (with embedding model)

We'll launch the server as a background subprocess in the notebook.


In [ ]:
# ── §5a Launch vLLM OpenAI Server ────────────────────────────────────────────
import requests

VLLM_PORT     = 8080
VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}/v1"

def start_vllm_server(model_id: str, port: int):
    """Start vLLM OpenAI-compatible server as a background process."""
    if not HAS_GPU:
        return None
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model",                   model_id,
        "--dtype",                   "bfloat16",
        "--max-model-len",           "4096",
        "--gpu-memory-utilization",  "0.85",
        "--port",                    str(port),
        "--host",                    "0.0.0.0",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"⏳ vLLM server starting on :{port}  (PID {proc.pid})")
    print("   Polling for readiness...")
    return proc

vllm_proc = start_vllm_server(MODEL_ID, VLLM_PORT)


In [ ]:
# ── §5b Wait for Server Ready ─────────────────────────────────────────────────
def wait_for_server(base_url: str, timeout_s: int = 120) -> bool:
    """Poll /v1/models until server responds or timeout."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            r = requests.get(f"{base_url}/models", timeout=2)
            if r.status_code == 200:
                model_id = r.json()["data"][0]["id"]
                print(f"\n✅ Server ready! Serving: {model_id}")
                return True
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(3)
    print("\n⚠️  Server startup timed out")
    return False

if HAS_GPU and vllm_proc:
    SERVER_READY = wait_for_server(VLLM_BASE_URL)
else:
    SERVER_READY = False
    print("⚠️  No GPU — server not started. OpenAI client calls will use stub mode.")


In [ ]:
# ── §5c Query with OpenAI Client (Drop-in Swap) ──────────────────────────────
# This is the KEY insight: same openai library, just different base_url.
# In real infra you'd set OPENAI_API_BASE env var — application code changes ZERO lines.

def query_self_hosted(prompt: str, system: str = "You are a concise technical expert.") -> str:
    """Query the vLLM server using the standard openai client."""
    if not SERVER_READY:
        return f"[STUB — no GPU] Would answer: {prompt[:60]}..."

    client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        max_tokens=200,
        temperature=0.7,
    )
    return resp.choices[0].message.content

# Demo
t0 = time.time()
answer = query_self_hosted("What is PagedAttention and why does it improve LLM serving throughput?")
elapsed = time.time() - t0

print(f"Response ({elapsed:.2f}s):")
print(answer)

# 💡 EXPERIMENT: Point this at a remote vLLM instance by changing VLLM_BASE_URL.
#    The code is identical whether the model runs locally or on a remote A100.


In [ ]:
# ── §5d Streaming — Same stream=True as OpenAI ───────────────────────────────
def stream_self_hosted(prompt: str) -> None:
    """Stream tokens as they arrive — identical to streaming from OpenAI API."""
    if not SERVER_READY:
        print("[STUB] Streaming: PagedAttention... treats... KV... cache... like... paged... memory.")
        return

    client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")
    print("▶ Streaming response: ", end="")

    with client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
        stream=True,    # ← same parameter as OpenAI
    ) as stream:
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                print(delta, end="", flush=True)
    print("\n✅ Stream complete")

stream_self_hosted("Explain the attention mechanism in 3 sentences, starting with the core intuition.")


## §6 Throughput Benchmark — vLLM vs HuggingFace Naive

Let's measure the actual speedup. We compare:

**Method A — HuggingFace sequential**: Load model with `transformers`, call `model.generate()` one prompt at a time.

**Method B — vLLM batch**: Send all prompts together. PagedAttention + continuous batching handles scheduling.

### What to expect on Colab T4
| Batch size | HF sequential | vLLM | Speedup |
|-----------|--------------|------|---------|
| 1 | ~120 tok/s | ~200 tok/s | ~1.7× |
| 8 | ~120 tok/s | ~800 tok/s | ~6.7× |
| 16 | ~120 tok/s | ~1,400 tok/s | ~12× |
| 32 | ~120 tok/s | ~2,000 tok/s | ~17× |

HF sequential throughput stays roughly constant (GPU processes one sequence at a time).  
vLLM throughput scales with batch size up to the VRAM limit.


In [ ]:
# ── §6 Throughput Benchmark ───────────────────────────────────────────────────
BENCH_PROMPTS = [
    f"Explain AI/ML concept #{i} in exactly 2 sentences." for i in range(16)
]

# ── Method A: vLLM batch ──────────────────────────────────────────────────────
def benchmark_vllm_batch(prompts: list) -> dict:
    if not HAS_GPU:
        return {"name": "vLLM batch", "elapsed_s": 0.9, "total_tokens": 800,
                "throughput_tps": 889.0, "stub": True}

    params = SamplingParams(temperature=0.0, max_tokens=50)
    t0 = time.time()
    outputs = llm.generate(prompts, params)
    elapsed = time.time() - t0

    total_tok = sum(len(o.outputs[0].token_ids) for o in outputs)
    return {"name": "vLLM batch", "elapsed_s": elapsed,
            "total_tokens": total_tok, "throughput_tps": total_tok / elapsed}

# ── Method B: HF sequential ───────────────────────────────────────────────────
def benchmark_hf_sequential(prompts: list) -> dict:
    if not HAS_GPU:
        return {"name": "HF sequential", "elapsed_s": 8.0, "total_tokens": 800,
                "throughput_tps": 100.0, "stub": True}

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print("  (Re-loading model weights for HF baseline...)", flush=True)
    tok   = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )
    model.eval()

    t0 = time.time()
    total_tok = 0
    with torch.no_grad():
        for p in prompts:
            inputs  = tok(p, return_tensors="pt").to(model.device)
            out     = model.generate(**inputs, max_new_tokens=50, do_sample=False,
                                     pad_token_id=tok.eos_token_id)
            total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    elapsed = time.time() - t0

    del model
    torch.cuda.empty_cache()
    return {"name": "HF sequential", "elapsed_s": elapsed,
            "total_tokens": total_tok, "throughput_tps": total_tok / elapsed}

# ── Run both benchmarks ───────────────────────────────────────────────────────
print(f"Benchmarking on {len(BENCH_PROMPTS)} prompts × 50 max tokens\n")

print("▶ vLLM batch...")
r_vllm = benchmark_vllm_batch(BENCH_PROMPTS)
print(f"  {r_vllm['throughput_tps']:.0f} tok/s  ({r_vllm['elapsed_s']:.2f}s total)\n")

print("▶ HF sequential...")
r_hf = benchmark_hf_sequential(BENCH_PROMPTS)
print(f"  {r_hf['throughput_tps']:.0f} tok/s  ({r_hf['elapsed_s']:.2f}s total)\n")

speedup = r_vllm["throughput_tps"] / r_hf["throughput_tps"]
print(f"🚀 vLLM speedup: {speedup:.1f}×  ({r_hf['elapsed_s']:.1f}s → {r_vllm['elapsed_s']:.1f}s)")
note = " [stub values — run with GPU for real numbers]" if r_vllm.get("stub") else ""
print(note)

# 💡 EXPERIMENT: Change len(BENCH_PROMPTS) to 32 and 64.
#    Plot throughput_tps vs batch_size for both methods.
#    The vLLM curve should be steeper — continuous batching scales with load.


## §7 Wiring Into the L31 Reliability Spine

Remember the L31 `FallbackChain`? A "tier" is just `Callable[[str], str]`.  
The chokepoint doesn't care what's inside — we plug in vLLM as Tier 0.

```
Tier 0: self-hosted vLLM    ← fastest, cheapest, on-prem
Tier 1: Anthropic Haiku     ← reliable cloud fallback
Tier 2: static response     ← always works, zero cost
```

**When should Tier 0 fail over to Tier 1?**
- vLLM server not running (cold start, crash)
- GPU OOM on a load spike
- Capability gap — routing complex reasoning tasks to Haiku/Sonnet
- Explicit confidence routing: "if self-hosted confidence < 0.8 → escalate"

**The model-agnostic insight from L31:**  
The circuit breaker, canary router, and cost meter don't need to know what's  
behind each tier. Swap Haiku for a self-hosted model → the harness adapts.


In [ ]:
# ── §7 Reliability Spine — Self-Hosted as Tier 0 ────────────────────────────
# Minimal FallbackChain reproduced from L31 — self-contained for this lesson.
from dataclasses import dataclass, field
from typing import Callable

@dataclass(frozen=True)
class FallbackResult:
    answer: str
    tier_used: str       # "self_hosted" | "haiku" | "static"
    latency_s: float
    ok: bool

@dataclass
class FallbackChain:
    """Tries tiers in order. First success wins; on failure falls to next."""
    tiers: list[tuple[str, Callable[[str], str]]]

    def call(self, prompt: str) -> FallbackResult:
        t0 = time.time()
        for name, fn in self.tiers:
            try:
                answer = fn(prompt)
                return FallbackResult(answer, name, time.time() - t0, ok=True)
            except Exception as e:
                print(f"  ⚡ {name} failed ({e.__class__.__name__}: {e}) → next tier")
        return FallbackResult(
            "I can't answer that reliably right now.",
            "static", time.time() - t0, ok=False
        )

# ── Define tiers ──────────────────────────────────────────────────────────────
def tier_self_hosted(prompt: str) -> str:
    if not SERVER_READY:
        raise RuntimeError("vLLM server not running")
    return query_self_hosted(prompt)

def tier_haiku(prompt: str) -> str:
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

def tier_static(prompt: str) -> str:
    return "I can't answer reliably right now. Please try again in a moment."

# ── Build the chain ───────────────────────────────────────────────────────────
chain = FallbackChain(tiers=[
    ("self_hosted", tier_self_hosted),  # Tier 0 — fastest, cheapest
    ("haiku",       tier_haiku),        # Tier 1 — reliable cloud
    ("static",      tier_static),       # Tier 2 — always succeeds
])

# ── Demo ──────────────────────────────────────────────────────────────────────
DEMO_QUESTIONS = [
    "What is the difference between BF16 and FP16?",
    "Why does continuous batching improve GPU utilization?",
]

print("=== Reliability Chain Demo ===\n")
for q in DEMO_QUESTIONS:
    result = chain.call(q)
    status = "✅" if result.ok else "⚠️"
    print(f"{status} [{result.tier_used}] ({result.latency_s:.2f}s)")
    print(f"   Q: {q}")
    print(f"   A: {result.answer[:120]}{'...' if len(result.answer) > 120 else ''}")
    print()

# 💡 EXPERIMENT: Kill the vLLM server (uncomment next line) and re-run.
#    Watch the chain seamlessly fall through to Haiku.
# if vllm_proc: vllm_proc.terminate()


In [ ]:
# ── §7b Cost Comparison Across Tiers ─────────────────────────────────────────
print(f"{'Tier':<25} {'$/1K tokens':>12}  {'Latency (ms)':>14}  {'Availability':>13}  Monthly @ 100M tok")
print("-" * 90)

tier_data = [
    ("self_hosted (T4)",  0.010,  60, 0.97),   # GPU uptime ~97%
    ("haiku API",         0.800, 400, 0.999),   # Anthropic SLA
    ("static fallback",   0.000,   1, 1.000),   # always on
]

MONTHLY_TOKENS = 100_000_000  # 100M tokens/month

for name, cost_per_k, latency_ms, avail in tier_data:
    monthly_cost = MONTHLY_TOKENS / 1000 * cost_per_k
    print(f"{name:<25} ${cost_per_k:>11.3f}  {latency_ms:>14}ms  {avail:>13.1%}  "
          f"${monthly_cost:>10,.2f}")

print()
print("Production strategy: route 90% → self_hosted, 10% overflow → haiku")
print(f"  Blended cost: {0.9*0.010 + 0.1*0.800:.3f} $/1K tokens")
print(f"  Monthly @ 100M tokens: ${(0.9*0.010 + 0.1*0.800) * 100_000_000 / 1000:,.2f}")
print(f"  vs all-Haiku:          ${0.800 * 100_000_000 / 1000:,.2f}")
print(f"  Savings: {(1 - (0.9*0.010 + 0.1*0.800)/0.800)*100:.0f}%")


## §8 Pitfalls — Things That Will Bite You

| # | Pitfall | Symptom | Fix |
|---|---------|---------|-----|
| 1 | **VRAM math ignores KV cache** | OOM on first long request despite weights fitting | Budget `weights + kv_cache(max_seq × batch)` before deploying |
| 2 | **Context length mismatch** | Outputs truncated mid-sentence | Set `--max-model-len` to match model's trained context window |
| 3 | **Tokenizer drift** | Self-hosted tokenizes `<tool>` tags differently than API model | Run `tokenizer(text).tokens` parity test vs reference |
| 4 | **Sampling param drift** | `temperature=0.7` on Qwen ≠ `temperature=0.7` on Claude (different sampling implementations) | Re-calibrate outputs empirically; don't assume numeric equivalence |
| 5 | **Cold start latency spike** | First request 3–5× slower (CUDA kernel JIT compilation) | Warm up server with a dummy request on startup |
| 6 | **No auth on vLLM server** | Anyone on the network uses your GPU | Add `--api-key <secret>` and put behind a reverse proxy (nginx/Caddy) |
| 7 | **HuggingFace Hub rate limits** | Model download fails silently after server starts | Pre-download with `huggingface-cli download <model>` before prod deploy |
| 8 | **BF16 on old GPUs** | Silently wrong results on V100 (no BF16 support) | Use `--dtype float16` on V100/older hardware; verify with `torch.cuda.is_bf16_supported()` |
| 9 | **Capability gap for complex tasks** | 1.5B model confidently hallucinates on multi-step reasoning | Route complex tasks (ReAct, planning, analysis) to Haiku/Sonnet; self-host for classification/summarization only |
| 10 | **Unsafe server shutdown** | Corrupted in-flight KV state, zombie processes | Always `proc.terminate()` then `proc.wait(timeout=10)`, never `proc.kill()` cold |

### Task routing decision rule

```python
SELF_HOSTED_TASKS = {"classify", "summarize", "extract", "translate", "embed"}
API_TASKS         = {"reason", "plan", "analyze", "synthesize", "code"}

tier = "self_hosted" if task_type in SELF_HOSTED_TASKS else "haiku"
```


In [ ]:
# ── §8b Graceful Server Shutdown ──────────────────────────────────────────────
def shutdown_vllm(proc) -> None:
    """Always call this before ending the session to free GPU memory."""
    if proc is None:
        return
    print(f"⏹ Terminating vLLM server (PID {proc.pid})...")
    proc.terminate()
    try:
        proc.wait(timeout=10)
        print("✅ Server shut down cleanly")
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
        print("⚠️  Force-killed after 10s timeout")

# Uncomment when done:
# shutdown_vllm(vllm_proc)

print("Call shutdown_vllm(vllm_proc) to release GPU memory when done with the notebook.")
print()

# Show server process info
if HAS_GPU and vllm_proc:
    print(f"vLLM server PID: {vllm_proc.pid}")
    print(f"Status: {'running' if vllm_proc.poll() is None else 'stopped'}")


## §9 Homework

1. **VRAM stress test**: change `gpu_memory_utilization` from 0.85 to 0.95 and fire 50 concurrent requests via `asyncio.gather`. Does vLLM OOM or gracefully queue? Compare to 0.80.

2. **Upgrade the model**: swap `Qwen2.5-1.5B` for `Qwen2.5-7B-Instruct-AWQ` (pre-quantized INT4, ~4 GB). Add `quantization="awq"` to the `LLM` constructor. Compare output quality on 5 prompts vs the 1.5B model.

3. **Complexity-based routing**: modify `FallbackChain` to route by prompt length — prompts under 100 chars go to `self_hosted`, longer ones to `haiku`. Measure cost savings vs quality difference on 20 test prompts.

4. **Benchmark at scale**: run the throughput benchmark for `N = [1, 4, 8, 16, 32, 64]` prompts. Plot `throughput (tok/s)` vs `N` for vLLM and HF on the same axes. The vLLM slope should be steeper — verify it is and annotate where the curve flattens (VRAM saturation).

5. **Production hardening**: add `--api-key $(python -c "import secrets; print(secrets.token_hex(16))")` to the server start command. Update `query_self_hosted` to pass the key as `api_key=` to the OpenAI client. This is minimum viable auth for a shared GPU box.

## Coming Up — L38: QLoRA Fine-Tuning on Real Data

You now have a vLLM server running `Qwen2.5-1.5B`. In L38 we **fine-tune** it to specialize on a domain.

The key insight: fine-tuning can close the gap between a 1.5B and a 7B model for narrow tasks. A 1.5B model fine-tuned on 500 domain-specific examples often beats a 7B base model on those exact tasks — at 5× lower serving cost.

**L38 topics**: dataset formatting (ChatML/Alpaca), LoRA rank/alpha trade-offs, QLoRA (4-bit base + FP16 adapters), `SFTTrainer`, loss curves, before/after benchmark.
